In [1]:
from pyflink.datastream import StreamExecutionEnvironment, KeyedProcessFunction, RuntimeContext
from pyflink.table import StreamTableEnvironment, EnvironmentSettings
from pyflink.common import Types, Configuration, Encoder, Row
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.datastream.connectors import FileSink, OutputFileConfig

In [2]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
t_env = StreamTableEnvironment.create(env, environment_settings=settings)

In [4]:
t_env.execute_sql("""
CREATE TABLE klines_source (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/klines/ADAUSDT/2025-09-27',
    'format' = 'csv'
)
""")

In [5]:
input_stream = (
    t_env.to_data_stream(t_env.from_path("klines_source"))
    .map(lambda r: Row("ADAUSDT", int(r[0].timestamp()), float(r[5])),
         output_type=Types.ROW([Types.STRING(), Types.INT(), Types.DOUBLE()]))
)

In [6]:
class EMA7Function(KeyedProcessFunction):
    def open(self, runtime_context: RuntimeContext):
        self.ema_state = runtime_context.get_state(
            ValueStateDescriptor("ema7", Types.DOUBLE())
        )
        self.buffer = []
        self.alpha = 2 / (7 + 1)

    def process_element(self, value, ctx):
        symbol, ts, close_price = value
        prev_ema = self.ema_state.value()

        if prev_ema is None:
            self.buffer.append(close_price)
            if len(self.buffer) < 7:
                return
            ema = sum(self.buffer) / len(self.buffer)
            self.buffer.clear()
        else:
            ema = self.alpha * close_price + (1 - self.alpha) * prev_ema

        self.ema_state.update(ema)
        yield Row(symbol, ts, close_price, ema)

In [7]:
ema_stream = (
    input_stream
    .key_by(lambda x: x[0])
    .process(
        EMA7Function(),
        output_type=Types.ROW([
            Types.STRING(), Types.INT(), Types.DOUBLE(), Types.DOUBLE()
        ])
    )
)

In [ ]:
t_env.execute_sql("DROP TABLE IF EXISTS ema7_sink")
t_env.execute_sql("""
CREATE TABLE ema7_sink (
    symbol STRING,
    ts INT,
    close_price DOUBLE,
    ema7 DOUBLE
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/ema7',
    'format' = 'csv'
)
""")

t_env.create_temporary_view("ema_stream", t_env.from_data_stream(ema_stream))
t_env.execute_sql("INSERT INTO ema7_sink SELECT * FROM ema_stream")

In [9]:
!jupyter nbconvert --to script test_transform_job_pattern_two.ipynb

I0000 00:00:1760419698.195361     202 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


[NbConvertApp] Converting notebook test_transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 2777 bytes to test_transform_job_pattern_two.py
